# MS3SEG VentiMorph-RelNet V2.7 -- five-fold aggregation

Reproduces the **VentiMorph-RelNet (ours)** row of **Table 7** and the
**VentiMorph-RelNet V2.7 (frozen calibrated)** row of **Table 8** from the per-fold
patient-level validation results.

**Protocol** (matches the paper): patient-level 3-D evaluation on each fold's 16 held-out
validation patients; the Fold-0 calibration is frozen and applied unchanged to Folds 1-4;
the locked 20-patient test set is never used.

**Fold numbering:** the lab sheet labels folds 1-5; internally these are CV folds 0-4.
`Fold-1` here == `fold 0` in the training / evaluation notebooks (its numbers come from
`ms3seg-ventimorph-relnet-v2-7-fold0-validation-eva.ipynb` /
`results/fold0_validation/overall_summary.csv`).

Metric = mean per-patient Dice for each class. `mean_fg` = mean(ventricle, nWMH, abWMH).


In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda v: f'{v:.6f}')

# where the generated table CSVs are written
OUT_DIR = 'results/five_fold' if os.path.isdir('results') else \
          ('../results/five_fold' if os.path.isdir('../results') else '.')
os.makedirs(OUT_DIR, exist_ok=True)
print('output dir:', OUT_DIR)


output dir: results/five_fold


## 1. Per-fold results -- VentiMorph-RelNet V2.7 (frozen calibrated)

Transcribed from the per-fold patient-level evaluation (lab sheet). Fold-1 is cross-checked
against the committed `results/fold0_validation/overall_summary.csv`.


In [2]:
ventimorph = pd.DataFrame({
    'fold_label': [1, 2, 3, 4, 5],          # lab-sheet numbering
    'cv_fold':    [0, 1, 2, 3, 4],          # notebook / split numbering
    'ventricle':  [0.851588, 0.836521, 0.850324, 0.855717, 0.851029],
    'nwmh':       [0.636120, 0.626910, 0.640466, 0.642706, 0.616330],
    'abwmh':      [0.777938, 0.753907, 0.744385, 0.789882, 0.767844],
    'mean_fg_recorded': [0.755216, 0.739136, 0.745058, 0.762588, 0.745067],
})

# recompute mean_fg from the three class values for internal consistency
ventimorph['mean_fg'] = ventimorph[['ventricle', 'nwmh', 'abwmh']].mean(axis=1)
ventimorph.to_csv(f'{OUT_DIR}/ventimorph_per_fold.csv', index=False)
ventimorph


,fold_label,cv_fold,ventricle,nwmh,abwmh,mean_fg_recorded,mean_fg
0,1,0,0.851588,0.636120,0.777938,0.755216,0.755215
1,2,1,0.836521,0.626910,0.753907,0.739136,0.739113
2,3,2,0.850324,0.640466,0.744385,0.745058,0.745058
3,4,3,0.855717,0.642706,0.789882,0.762588,0.762768
4,5,4,0.851029,0.616330,0.767844,0.745067,0.745068


In [3]:
# sanity: Fold-1 must match the committed Fold-0 evaluation summary
expected_fold0 = dict(ventricle=0.851588, nwmh=0.636120, abwmh=0.777938, mean_fg=0.755216)
row0 = ventimorph.loc[ventimorph.cv_fold == 0].iloc[0]
for k, v in expected_fold0.items():
    diff = abs(row0[k] - v)
    print(f'{k:10s} sheet={row0[k]:.6f}  eval-csv={v:.6f}  |diff|={diff:.2e}  {"OK" if diff < 5e-4 else "CHECK"}')


ventricle  sheet=0.851588  eval-csv=0.851588  |diff|=0.00e+00  OK
nwmh       sheet=0.636120  eval-csv=0.636120  |diff|=0.00e+00  OK
abwmh      sheet=0.777938  eval-csv=0.777938  |diff|=0.00e+00  OK
mean_fg    sheet=0.755215  eval-csv=0.755216  |diff|=6.67e-07  OK


## 2. Per-fold results -- capacity-matched baselines (Table 8)

Paste your per-fold U-Net and U-Net++ patient-level Dice below (same 5 folds, same split).
If you leave a model as `None`, the notebook falls back to the published Table-8 mean/std for
that row and prints a note.


In [4]:
# --- FILL IN: one value per fold, order = folds 1..5 (cv_fold 0..4) ---
unet_per_fold = {
    # 'ventricle': [..., ..., ..., ..., ...],
    # 'nwmh':      [..., ..., ..., ..., ...],
    # 'abwmh':     [..., ..., ..., ..., ...],
}

unetpp_per_fold = {
    # 'ventricle': [..., ..., ..., ..., ...],
    # 'nwmh':      [..., ..., ..., ..., ...],
    # 'abwmh':     [..., ..., ..., ..., ...],
}

# Published Table-8 values (used only as fallback / cross-check)
published_table8 = {
    'Capacity-matched U-Net':   dict(params_m=5.946,
        ventricle=(0.9049, 0.0061), nwmh=(0.7020, 0.0082),
        abwmh=(0.7794, 0.0304), mean_fg=(0.7954, 0.0099)),
    'Capacity-matched U-Net++': dict(params_m=6.050,
        ventricle=(0.9094, 0.0044), nwmh=(0.7101, 0.0113),
        abwmh=(0.7889, 0.0298), mean_fg=(0.8028, 0.0064)),
    'VentiMorph-RelNet V2.7 (frozen calibrated)': dict(params_m=5.97,
        ventricle=(0.8489, 0.0062), nwmh=(0.6325, 0.0117),
        abwmh=(0.76679, 0.0338), mean_fg=(0.7494, 0.0138)),
}

def per_fold_df(d):
    if not d or 'ventricle' not in d:
        return None
    df = pd.DataFrame({k: d[k] for k in ['ventricle', 'nwmh', 'abwmh']})
    df.insert(0, 'cv_fold', range(len(df)))
    df['mean_fg'] = df[['ventricle', 'nwmh', 'abwmh']].mean(axis=1)
    return df

unet_df = per_fold_df(unet_per_fold)
unetpp_df = per_fold_df(unetpp_per_fold)
print('U-Net per-fold provided   :', unet_df is not None)
print('U-Net++ per-fold provided  :', unetpp_df is not None)


U-Net per-fold provided   : False
U-Net++ per-fold provided  : False


## 3. Aggregate across the five folds

`mean` and sample standard deviation (`ddof=1`) across the five fold-level patient means,
plus the population SD (`ddof=0`) for reference.


In [5]:
CLASSES = ['ventricle', 'nwmh', 'abwmh', 'mean_fg']

def aggregate(df, name):
    out = {'model': name, 'folds': len(df)}
    for c in CLASSES:
        out[f'{c}_mean']     = df[c].mean()
        out[f'{c}_std']      = df[c].std(ddof=1)
        out[f'{c}_std_pop']  = df[c].std(ddof=0)
    return out

agg_rows = [aggregate(ventimorph, 'VentiMorph-RelNet V2.7 (frozen calibrated)')]
if unet_df is not None:
    agg_rows.append(aggregate(unet_df, 'Capacity-matched U-Net'))
if unetpp_df is not None:
    agg_rows.append(aggregate(unetpp_df, 'Capacity-matched U-Net++'))

agg = pd.DataFrame(agg_rows).set_index('model')
agg.filter(regex='_mean$|folds')


,folds,ventricle_mean,nwmh_mean,abwmh_mean,mean_fg_mean
model,,,,,
VentiMorph-RelNet V2.7 (frozen calibrated),5,0.849036,0.632506,0.766791,0.749444


In [6]:
# mean +/- sd, formatted
def fmt(df, name):
    r = df.loc[name]
    return {c: f"{r[f'{c}_mean']:.4f} +/- {r[f'{c}_std']:.4f}" for c in CLASSES}

pd.DataFrame({m: fmt(agg, m) for m in agg.index}).T


,ventricle,nwmh,abwmh,mean_fg
VentiMorph-RelNet V2.7 (frozen calibrated),0.8490 +/- 0.0073,0.6325 +/- 0.0109,0.7668 +/- 0.0182,0.7494 +/- 0.0094


## 4. Table 7 -- numerical comparison with published MS3SEG models

Other methods' numbers are taken verbatim from the MS3SEG paper (ref [8], FLAIR-only);
only the **VentiMorph-RelNet (ours)** row is computed here.


In [7]:
v = agg.loc['VentiMorph-RelNet V2.7 (frozen calibrated)']

table7 = pd.DataFrame([
    ['U-Net [8]',       'FLAIR',            0.8897, 0.6452, 0.6686, 0.7345, 'n/r'],
    ['U-Net++ [8]',     'FLAIR',            0.8934, 0.6275, 0.6634, 0.7281, 'n/r'],
    ['UNETR [8]',       'FLAIR',            0.8240, 0.5245, 0.5551, 0.6345, 'n/r'],
    ['Swin UNETR [8]',  'FLAIR',            0.8632, 0.5591, 0.5886, 0.6703, 'n/r'],
    ['VentiMorph-RelNet (ours)', 'FLAIR+T1+T2, 2.5D',
        round(v['ventricle_mean'], 4), round(v['nwmh_mean'], 4),
        round(v['abwmh_mean'], 5),  round(v['mean_fg_mean'], 4), '5.97 M'],
], columns=['Method', 'Input', 'Ventricle', 'nWMH', 'abWMH', 'Mean FG', 'Parameters'])

table7.to_csv(f'{OUT_DIR}/table7_numerical_comparison.csv', index=False)
table7


,Method,Input,Ventricle,nWMH,abWMH,Mean FG,Parameters
0,U-Net [8],FLAIR,0.889700,0.645200,0.668600,0.734500,n/r
1,U-Net++ [8],FLAIR,0.893400,0.627500,0.663400,0.728100,n/r
2,UNETR [8],FLAIR,0.824000,0.524500,0.555100,0.634500,n/r
3,Swin UNETR [8],FLAIR,0.863200,0.559100,0.588600,0.670300,n/r
4,VentiMorph-RelNet (ours),"FLAIR+T1+T2, 2.5D",0.849000,0.632500,0.766790,0.749400,5.97 M


## 5. Table 8 -- controlled five-fold architecture ablation

mean +/- sd across the five fold-level patient means. Rows with per-fold data provided in
section 2 are computed; otherwise the published values are shown (flagged).


In [8]:
def table8_row(name):
    if name in agg.index:
        r = agg.loc[name]
        src = 'computed'
        cells_ = {c: f"{r[f'{c}_mean']:.4f} +/- {r[f'{c}_std']:.4f}" for c in CLASSES}
        params = published_table8[name]['params_m']
    else:
        p = published_table8[name]
        src = 'published (no per-fold data supplied)'
        cells_ = {c: f"{p[c][0]:.4f} +/- {p[c][1]:.4f}" for c in CLASSES}
        params = p['params_m']
    return {'Architecture': name, 'Folds': 5, 'Params (M)': params,
            'Ventricle': cells_['ventricle'], 'nWMH': cells_['nwmh'],
            'abWMH': cells_['abwmh'], 'Mean FG': cells_['mean_fg'], 'source': src}

table8 = pd.DataFrame([
    table8_row('Capacity-matched U-Net'),
    table8_row('Capacity-matched U-Net++'),
    table8_row('VentiMorph-RelNet V2.7 (frozen calibrated)'),
])
table8.to_csv(f'{OUT_DIR}/table8_five_fold_ablation.csv', index=False)
table8


,Architecture,Folds,Params (M),Ventricle,nWMH,abWMH,Mean FG,source
0,Capacity-matched U-Net,5,5.946000,0.9049 +/- 0.0061,0.7020 +/- 0.0082,0.7794 +/- 0.0304,0.7954 +/- 0.0099,published (no per-fold data supplied)
1,Capacity-matched U-Net++,5,6.050000,0.9094 +/- 0.0044,0.7101 +/- 0.0113,0.7889 +/- 0.0298,0.8028 +/- 0.0064,published (no per-fold data supplied)
2,VentiMorph-RelNet V2.7 (frozen calibrated),5,5.970000,0.8490 +/- 0.0073,0.6325 +/- 0.0109,0.7668 +/- 0.0182,0.7494 +/- 0.0094,computed


## 6. Cross-check against the published paper values


In [9]:
paper = published_table8['VentiMorph-RelNet V2.7 (frozen calibrated)']
print(f"{'metric':10s} {'computed mean':>14s} {'paper mean':>12s} {'d':>9s}   "
      f"{'computed sd':>12s} {'paper sd':>10s}")
for c in CLASSES:
    cm = agg.loc['VentiMorph-RelNet V2.7 (frozen calibrated)', f'{c}_mean']
    cs = agg.loc['VentiMorph-RelNet V2.7 (frozen calibrated)', f'{c}_std']
    pm, ps = paper[c]
    flag = 'OK' if abs(cm - pm) < 1e-3 else 'CHECK'
    print(f'{c:10s} {cm:14.5f} {pm:12.5f} {cm - pm:+9.5f}   {cs:12.5f} {ps:10.5f}  {flag}')


metric      computed mean   paper mean         d    computed sd   paper sd
ventricle         0.84904      0.84890  +0.00014        0.00730    0.00620  OK
nwmh              0.63251      0.63250  +0.00001        0.01088    0.01170  OK
abwmh             0.76679      0.76679  +0.00000        0.01821    0.03380  OK
mean_fg           0.74944      0.74940  +0.00004        0.00943    0.01380  OK


### Notes

* **Means reproduce the paper exactly** (to 4-5 dp): Ventricle ~0.8490, nWMH 0.6325,
  abWMH 0.76679, Mean FG 0.7494.
* The **sample SD of these five fold means is smaller than the +/- printed in Table 8**
  (most visibly abWMH: ~0.018 here vs 0.0338 in the paper). If Table 8's +/- is intended to be
  the SD across the five fold-level patient means, either the per-fold abWMH values in
  section 1 need a digit check, or the paper's dispersion is a different quantity
  (e.g. the pooled per-patient SD over all 80 validation patients, which for Fold-0 alone is
  already 0.089 for abWMH -- see `results/fold0_validation/class_summary.csv`). Confirm which
  definition the manuscript uses and adjust `agg[...]_std` / the Table-8 text accordingly.
* Ventricle mean here is 0.8490; the paper prints 0.8489. The ~0.0003 gap is transcription
  noise in the handwritten ventricle column -- fix the exact per-fold digits in section 1 if
  an exact match is required.
